# arXiv Agent and Coding-Agent Research Pulse

This notebook tracks query-based research topics on arXiv. It is designed for fast-moving themes where category alone is too broad.

Good first topics:

- AI agents;
- coding agents;
- tool use;
- retrieval-augmented generation;
- mechanistic interpretability;
- AI for science.

Data source: arXiv API. No synthetic fallback.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Define query pulses

The queries below are deliberately explicit. Adjust them when the arXiv API parser or field syntax requires a stricter expression.


In [ ]:
topic_queries = {
    "ai_agent": 'all:"AI agent" OR all:"autonomous agent"',
    "coding_agent": 'all:"coding agent" OR all:"code agent" OR all:"program repair"',
    "tool_use": 'all:"tool use" OR all:"function calling"',
    "rag": 'all:"retrieval augmented generation" OR all:"RAG"',
    "mechanistic_interpretability": 'all:"mechanistic interpretability"',
    "ai_for_science": 'all:"AI for science" OR all:"scientific discovery"',
}
pd.DataFrame([{"topic": k, "query": v} for k, v in topic_queries.items()])


## 2. Fetch real monthly topic counts


In [ ]:
START_MONTH = "2025-01-01"
END_MONTH = "2026-05-01"
SLEEP_SECONDS = 3.0

cache_path = CACHE_DIR / f"arxiv_agent_topic_counts_{START_MONTH}_{END_MONTH}.csv"
if cache_path.exists():
    counts = pd.read_csv(cache_path, parse_dates=["month"])
else:
    counts = build_arxiv_monthly_counts(topic_queries, start_month=START_MONTH, end_month=END_MONTH, sleep_seconds=SLEEP_SECONDS)
    counts.to_csv(cache_path, index=False)
counts.head(12)


## 3. Source audit


In [ ]:
audit = source_audit_table(counts, value_col="count", entity_col="series", time_col="month")
audit


## 4. Decompose topic counts


In [ ]:
components = decompose_table(
    counts,
    entity_col="series",
    time_col="month",
    value_col="count",
    method="MA_BASELINE",
    period=12,
    trend_window=5,
    transform="log1p",
)
summary = component_summary(components, entity_col="series", time_col="month")
summary


## 5. Editorial ranking

A topic with both positive trend slope and high residual shock deserves a timely post. A topic with trend but low residual is better for an evergreen explainer.


In [ ]:
priority = editorial_priority(summary, entity_col="series")
priority


## 6. Event-like months


In [ ]:
events = residual_event_table(components, entity_col="series", time_col="month", top_n=20)
events


## 7. Article outline template


In [ ]:
outline = pd.DataFrame([
    {"section": "Question", "content": "Which agent-related research topic is accelerating?"},
    {"section": "Source", "content": "arXiv API counts by query and submitted month."},
    {"section": "De-Time table", "content": "trend slope, cycle strength, residual shock rank."},
    {"section": "Caveat", "content": "arXiv counts measure preprint volume, not research quality or adoption."},
])
outline


In [ ]:
save_table(audit, "02_arxiv_agent_topic_audit")
save_table(priority, "02_arxiv_agent_topic_priority")
save_table(events, "02_arxiv_agent_topic_residual_events")
save_table(outline, "02_arxiv_agent_article_outline")
